<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day20_practice2_JSON_%EA%B5%AC%EC%A1%B0%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# JSON 구조화
# LLM 은 '말이 많다', 서버는 '정확한 JSON'만 사용.
# JSON 을 강제하는 프롬포트, 그래도 지저분한 응답을 추출 → 파싱 → 검증 → 보정


In [18]:
import json, re

In [19]:
# 셀 1. JSON 강제 프롬포트
JSON_PROMPT = """ 다음 도로 위험 신고를 분석해 JSON 으로만 응답하라.
[스키마]
{"hazard_type": "포트홀|균열|맨홀|낙하물|기타",
  "severity": 1-5 점수,
  "location": "문장에서 추출한 위치 (없으면 null)",
  "summary": "한 줄 요약"}
[규칙] JSON 외의 어떤 텍스트도 출력하지 마라, 코드블록 금지
[신고] {신고문}
"""
print("JSON 강제 프롬포트의 3요소: 스키마 명시 + '만' 강조 + 금지 목록")

JSON 강제 프롬포트의 3요소: 스키마 명시 + '만' 강조 + 금지 목록


In [20]:
# 셀 2. 실전 4대 사고 - 진짜 LLM 응답은 이렇게 온다
# 아래는 실제 서비스에서 흔히 만나는 응답 유형
응답_사례 = {
    "1) 모범 응답": '{"hazard_type": "포트홀", "severity": 4, "location": "강남대로 사거리", "summary": "사거리 대형 포트홀"}',

    "2) 코드펜스": '''```json
{"hazard_type": "포트홀", "severity": 4, "location": "강남대로 사거리", "summary": "사거리 대형 포트홀"}
```''',

    "3) 사족 문장": '''네, 분석해드리겠습니다!
{"hazard_type": "포트홀", "severity": 4, "location": "강남대로 사거리", "summary": "사거리 대형 포트홀"}
추가 문의가 있으시면 말씀해주세요.''',

    "4) 타입·키 오염": '{"hazard_type": "pothole", "severity": "4점", "summary": "사거리 대형 포트홀", "confidence": 0.9}',
}

In [21]:
for name, r in 응답_사례.items():
  try:
    json.loads(r) # JSON 형식의 문자열을 파이썬 객체로 변환
    status = "json.loads OK"
  except json.JSONDecodeError: # 내용이 유효한 JSON 형식이 아니면
    status = "json.loads 실패!"
  print(f"{name}: {status}")

1) 모범 응답: json.loads OK
2) 코드펜스: json.loads 실패!
3) 사족 문장: json.loads 실패!
4) 타입·키 오염: json.loads OK


In [22]:
# 셀 3. 1단계 - JSON '추출':  속에서 JSON 만 건진다. 사족과 포드펜스 제거.
def extract_json(text):
  text = re.sub(r"```(?:json)?", "", text) # 코드펜스 제거, re.sub(패턴, 바꿀 문자열, 대상)
  start = text.find("{")
  if start == -1:
    return None
  depth = 0
  for i in range(start, len(text)):
    if text[i] == "{": # 여는 중괄호를 만나면 겹이 하나 늘어남
      depth += 1      # 열리면 +1
    elif text[i] == "}":
      depth -= 1      # 닫히면 -1
      if depth == 0:  # 완결 중괄호가 열고 닫힘.
        try:
          return json.loads(text[start:i+1])    # 첫 '{' 부터 짝이 맞는 '}' 까지만 잘라서 변환, json → dict 변환
        except json.JSONDecodeError:
          return None
  return None

print("\n[추출기 테스트]")
for name, r in 응답_사례.items():
  result = extract_json(r)
  print(f"{name}: {'추출 성공' if result else '실패'}")


[추출기 테스트]
1) 모범 응답: 추출 성공
2) 코드펜스: 추출 성공
3) 사족 문장: 추출 성공
4) 타입·키 오염: 추출 성공


In [24]:
# 셀 4. 2단계 - _normalize: 검증하고 보정
# 서버의 출력: 키 4개, 타입 고정
VALID_TYPES = {"포트홀", "균열", "맨홀", "낙하물", "기타"}
TYPE_ALIAS = {"pothole": "포트홀", "crack": "균열", "manhole": "맨홀"}

def _normalize(data):
  if not isinstance(data, dict): # isinstance(값, 타입) → True
    return None
  out = {}
  t = str(data.get("hazard_type", "")).strip().lower()
  t = TYPE_ALIAS.get(t, t) # 영문이면 한글로 보정
  out["hazard_type"] = t if t in VALID_TYPES else "기타"

  sev = re.search(r"\d+", str(data.get("severity", ""))) # re.search(패턴, 문자열) 문자열에서 패턴(점수 하나이상)찾기, re.Match 객체(문자열"4") 또는 None
  sev = int(sev.group()) if sev else 3 # sev.group(): 일치한 문자열 전체를 int로 변환, 없으면 기본값 3

  out["severity"] = min(max(sev, 1), 5) # 범위 1-5 조정

  loc = data.get("location")
  out["location"] = loc if isinstance(loc, str) and loc.strip() else None

  out["summary"] = str(data.get("summary", "")).strip() or "요약 없음"

  return out

print("\n[_normalize 테스트: 4) 오염 응답] ")
dirty = extract_json(응답_사례["4) 타입·키 오염"])
clean = _normalize(dirty)
print("   오염:", dirty)
print("   보정:", clean)

# 검증
assert set(clean.keys()) == {"hazard_type", "severity", "location", "summary"}
assert clean["hazard_type"] in VALID_TYPES
assert isinstance(clean["severity"], int) and 1 <= clean["severity"] <= 5
print('   검증 통과: 영문 → 한글, "4점 → 4, 잉여 키 제거, 누락 키 폴백')


[_normalize 테스트: 4) 오염 응답] 
   오염: {'hazard_type': 'pothole', 'severity': '4점', 'summary': '사거리 대형 포트홀', 'confidence': 0.9}
   보정: {'hazard_type': '포트홀', 'severity': 4, 'location': None, 'summary': '사거리 대형 포트홀'}
   검증 통과: 영문 → 한글, "4점 → 4, 잉여 키 제거, 누락 키 폴백


In [26]:
# 셀 5. 전체 파이프라인
def parse_lim_response(text):
  data = extract_json(text) # 1단계: 추출
  return _normalize(data) if data else None # 2단계: 보정


print("\n[전체 파이프라인 : 4대 사고 일괄 처리]")
for name, r in 응답_사례.items():
  print(f" {name}: {parse_lim_response(r)}")

완전실패 = "죄송합니다. 신고 내용을 이해하지 못했습니다. "
print(f" 5) JSON 없음: {parse_lim_response(완전실패)}")


[전체 파이프라인 : 4대 사고 일괄 처리]
 1) 모범 응답: {'hazard_type': '포트홀', 'severity': 4, 'location': '강남대로 사거리', 'summary': '사거리 대형 포트홀'}
 2) 코드펜스: {'hazard_type': '포트홀', 'severity': 4, 'location': '강남대로 사거리', 'summary': '사거리 대형 포트홀'}
 3) 사족 문장: {'hazard_type': '포트홀', 'severity': 4, 'location': '강남대로 사거리', 'summary': '사거리 대형 포트홀'}
 4) 타입·키 오염: {'hazard_type': '포트홀', 'severity': 4, 'location': None, 'summary': '사거리 대형 포트홀'}
 5) JSON 없음: None
